This notebook was used to create 1:1 entity ratio for Upper and Lower Tier and create controlled dataset

### Diversity Test

Same number of lower tier entities as the top tier entities
Same number of qa pairs

In [ ]:
import json
import os

top_selected_entities = {}
lower_selected_entities = {}

CATEGORY="actors"

for root, _, files in os.walk(f"entity_facts_audited/{CATEGORY}/upper/"):
    for file in files:
        with open(os.path.join(root, file), 'r') as fp:
            top_selected_entities[' '.join(file.split('_')[:-1]).title()] = json.load(fp)

for root, _, files in os.walk(f"entity_facts_audited/{CATEGORY}/lower/"):
    for file in files:
        with open(os.path.join(root, file), 'r') as fp:
            lower_selected_entities[' '.join(file.split('_')[:2]).title()] = json.load(fp)

print("Top count: ", len(top_selected_entities))
print("Lower count: ", len(lower_selected_entities))

In [ ]:
# # Only for Actors
# # Exception Dicaprio -> DiCaprio

# top_selected_entities["Leonardo DiCaprio"] = top_selected_entities["Leonardo Dicaprio"]
# del top_selected_entities["Leonardo Dicaprio"]

In [ ]:
with open(f"final_dataset_metadata/{CATEGORY}/{CATEGORY}_entities_final.json", 'r') as fp:
    tech_entities = json.load(fp)

top_entities_ordered = []
lower_entities_ordered = []

for entity in tech_entities:
    if entity in top_selected_entities:
        top_entities_ordered.append(entity)

for entity in reversed(tech_entities):
    if entity in lower_selected_entities:
        lower_entities_ordered.append(entity)
        if len(lower_entities_ordered) == len(top_entities_ordered):
            break

In [ ]:

# replaced_entities = [
#     "Jackson Brundage",
#     "Michael Durrell",
#     "Joe Arquette"
# ]

# for i, e in enumerate(reversed(replaced_entities)):
#     lower_entities_ordered[-(i+1)] = e

# Sports
lower_entities_ordered.pop(0)
lower_entities_ordered.append("Luciano Vitullo")

lower_entities_ordered.pop(6)
lower_entities_ordered.append("Marcelo Zormann")

In [ ]:
lower_entities_ordered

In [ ]:
print("Top count: ", len(top_entities_ordered))
print("Lower count: ", len(lower_entities_ordered))

In [ ]:
for entity in lower_entities_ordered:
    with open(f"entity_facts_audited/{CATEGORY}/lower/{'_'.join(entity.lower().split(' '))}_audited.json", 'r') as fp:
        d = json.load(fp)
    print(entity, len(d))

In [ ]:
with open(f"final_dataset_metadata/{CATEGORY}/{CATEGORY}_entity_gender_map.json", 'r') as fp:
    entity_gender_map = json.load(fp)

In [ ]:
from generate_qa_with_era import generate_qa

lower_qa_pairs = []

for target_entity in lower_entities_ordered:
    distractors = [e for e in lower_entities_ordered if e != target_entity]

    lower_qa_pairs.extend(generate_qa(target_entity, distractors, lower_selected_entities, entity_gender_map))

In [ ]:
len(lower_qa_pairs)

In [ ]:
with open(f"final_dataset_v3/{CATEGORY}/new_{CATEGORY}_controlled_lower_qa.json", 'w') as fp:
    json.dump(lower_qa_pairs, fp, indent=4)

In [ ]:
all_top_qa_pairs = {}

for target_entity in top_entities_ordered:
    distractors = [e for e in top_entities_ordered if e != target_entity]

    all_top_qa_pairs[target_entity] = generate_qa(target_entity, distractors, top_selected_entities, entity_gender_map)

In [ ]:
{e: len(qa) for e, qa in all_top_qa_pairs.items()}

In [ ]:
import random

top_qa_pairs = []

k = len(lower_qa_pairs) // len(top_entities_ordered)
print(k)

for e, qa in all_top_qa_pairs.items():
    if len(qa) < k:
        top_qa_pairs.extend(qa)
        all_top_qa_pairs[e] = []
    else:
        chosen_pairs = random.sample(qa, k=k)
        top_qa_pairs.extend(chosen_pairs)
        selected_ids = {id(obj) for obj in chosen_pairs}
        all_top_qa_pairs[e] = [q for q in qa if id(q) not in selected_ids]

if len(top_qa_pairs) < len(lower_qa_pairs):
    while len(top_qa_pairs) < len(lower_qa_pairs):
        entity = random.choice(top_entities_ordered)
        if len(all_top_qa_pairs[entity]) == 0:
            continue
        chosen_pair = random.choice(all_top_qa_pairs[entity])
        top_qa_pairs.append(chosen_pair)
        all_top_qa_pairs[entity] = [q for q in all_top_qa_pairs[entity] if id(q) != id(chosen_pair)]
else:
    from collections import defaultdict
    per_entity_qa = defaultdict(list)
    for qa in top_qa_pairs:
        target_entity = qa["metadata"][qa["answer"]]
        per_entity_qa[target_entity].append(qa)
    while len(top_qa_pairs) > len(lower_qa_pairs):
        max_qa_count = max(per_entity_qa_count.values())
        max_qa_entities = [e for e, qa in per_entity_qa.items() if len(qa) == max_qa_count]
        entity = random.choice(max_qa_entities)
        selected_qa = random.choice(per_entity_qa[entity])
        per_entity_qa[entity] = [qa for qa in per_entity_qa[entity] if id(selected_qa) != id(qa)]
        top_qa_pairs = [qa for qa in top_qa_pairs if id(qa) != id(selected_qa)]

In [ ]:
len(top_qa_pairs)

In [ ]:
import json
from collections import defaultdict
# with open("actors_controlled_top_qa.json", 'r') as fp:
    # data = json.load(fp)
data = top_qa_pairs
print("Total: ", len(data))

per_entity_qa = defaultdict(list)
for qa in data:
    per_entity_qa[qa["metadata"][qa["answer"]]].append(qa)

{e: len(qa) for e, qa in per_entity_qa.items()}

In [ ]:
with open(f"final_dataset_v3/{CATEGORY}/new_{CATEGORY}_controlled_upper_qa.json", 'w') as fp:
    json.dump(top_qa_pairs, fp, indent=4)